# Class 9: Python API programming

In this class we learn how Python programs can talk to web services: how to request data from an API, and how to work with the JSON data that comes back.

## Learning outcomes

At the completion of this unit students should be able to:
1. Explain how a web request and response work, and what an API is.
2. Request JSON data from a public API using the `requests` library.
3. Explain why API providers require an API key.
4. Convert between JSON strings and Python objects using the `json` library.
5. Access values inside nested JSON structures.


## 9.1 API request and response handling

When you visit a website, you are sending a request to the server to access the page you are visiting. The server responds to your request by providing your web browser with an HTML file.

A very simple web page would contain HTML code like this:

```
<html>
  <head>
    <title>I am a very simple web page</title>
  </head>
  <body>
    This is the stuff in the body of this web page.
  </body>
</html>
```

So, the structure of the data that is returned in the response is HTML, CSS and JavaScript. The request that you send here is called the HTTP request (note that you start the web address with "http://").

Another way of requesting information on the web is via the API. The API (application programming interface) lets you request data, instead of requesting a page. There are several formats for the API data, but we will focus here on the JSON format for two reasons: it is very popular, and it resembles the Python `dictionary` data structure.

Let's have a look at an example website that lets you obtain the International Space Station astronauts in a JSON string.

http://api.open-notify.org/astros

That's an example of a JSON string, it looks exactly like a `dictionary`. The `people` key in the string is a `list` of dictionaries.

JSON data is built up from a small set of ingredients: *objects* (key-value pairs, like Python `dict`s), *arrays* (ordered lists, like Python `list`s), strings, numbers, booleans (`true`/`false`) and `null`. Objects and arrays can be nested inside each other.

We can access an API in Python via the `requests` library. Here is the code to do that.

**Note:** the code cells in this class need internet access, because they talk to live public APIs.

Let's request it from Python using the `requests` library. The `response` object holds what the server sent back, and its `.json()` method converts the JSON string into a Python dictionary:

In [1]:
import requests

response = requests.get("http://api.open-notify.org/astros.json")
data = response.json()
print(type(data))
print('Number of people in space:', data['number'])
print('First person:', data['people'][0])

<class 'dict'>
Number of people in space: 12
First person: {'craft': 'ISS', 'name': 'Oleg Kononenko'}


Here is another free API that needs no key: a list of public bike-share networks around the world. The list is long, so we print just a small part of it (the JSON data is simply a Python dictionary of nested lists and dictionaries — think how you would loop over it and answer questions like "how many bike networks does Australia have?").

In [2]:
import requests

response = requests.get("http://api.citybik.es/v2/networks")
data = response.json()
print(type(data))
print('Number of networks:', len(data['networks']))
print('One of them:', data['networks'][0]['name'], '-', data['networks'][0]['location']['city'])

<class 'dict'>
Number of networks: 815
One of them: Abu Dhabi Careem BIKE - Abu Dhabi


Now let's have a look at one more API.

Check this out: https://openweathermap.org/current

This page documents how you can use the OpenWeather API to access the current weather in a particular city. You send your request to an address like

```
http://api.openweathermap.org/data/2.5/weather?q=London
```

Wait — that is just an address, no key. Let's see what happens when we request it for Melbourne using `requests`:

In [3]:
site = 'http://api.openweathermap.org/data/2.5/weather?q=Melbourne'
response = requests.get(site)
data = response.json()
print(data)

{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


A `401 Unauthorised` response — the server answered, but it refused to give us the data without an **API key**.

Why is that? An API runs on a server and serves the data requirements of many internet users. The number of those users could be thousands, or even millions. Imagine how many people are accessing the Google Directions API every minute!

Because of that, providers of APIs only allow access to their APIs via an API key. This key is usually a large complex string, like `439d4b804bc8187953eb36d2a8c26a02`.

API providers will then restrict the access of the API service to a specific number of requests per some time interval. Above the quota, one needs to pay for access. As an example of how API's are priced, have a look at this page:

https://openweathermap.org/price

To access OpenWeather yourself, create a free account here: https://home.openweathermap.org/users/sign_up — then sign in, get your API key, and paste it into the code below in place of `'PASTE_YOUR_API_KEY_HERE'`. (Until you do, the request will keep returning the same `401` message.)

In [4]:
APIKey = 'PASTE_YOUR_API_KEY_HERE'
site = 'http://api.openweathermap.org/data/2.5/weather?q=Melbourne&units=metric'
response = requests.get(site + '&appid=' + APIKey)
data = response.json()
print(data)

{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


It's working now.

**GOTO Lab exercises 1**

## 9.2 The `json` library

Python allows you to easily access the content of a JSON string using the `json` library. It simply converts a JSON string in a python `dictionary`. For example:

In [5]:
import json

# (pretend we downloaded this JSON attendance record from an API)
json_text = '{ "Number of courses": 5, "Number of teachers": 10, "attendance": [ {"CourseName": "Course 1", "Attendance": 10}, {"CourseName": "Course 2", "Attendance": 13}, {"CourseName": "Course 3", "Attendance": 5} ] }'

d = json.loads(json_text)
print(d)
print('Attendance of Course 2:', d['attendance'][1]['Attendance'])

{'Number of courses': 5, 'Number of teachers': 10, 'attendance': [{'CourseName': 'Course 1', 'Attendance': 10}, {'CourseName': 'Course 2', 'Attendance': 13}, {'CourseName': 'Course 3', 'Attendance': 5}]}
Attendance of Course 2: 13


In [6]:
import json

x =  '{ "name":"Pi" }'

x = json.loads(x)

print(x["name"])

Pi


Let's see how the `json` library can help in accessing more complex JSON structures.





In [7]:
import json

x =  '{ "product":"PC" , "price": 1000 , "components": [{"name":"CPU","brand":"Intel"},{"name":"Hardisk","brand":"Toshiba"}] }'

x = json.loads(x)

print(x["product"])

print(x["components"][0])

print(x["components"][0]["name"])

PC
{'name': 'CPU', 'brand': 'Intel'}
CPU


In [8]:
import json
import os

d = {'name': 'Pi'}

f = open('pi.json', 'w')
json.dump(d, f)
f.close()

f = open('pi.json', 'r')
print(json.load(f))
f.close()

os.remove('pi.json')

{'name': 'Pi'}


**GOTO Lab exercise 2**

## Summary

- An *API* lets your program request data (usually JSON) from a server; `requests.get(url)` fetches it and `response.json()` converts the reply into Python dictionaries and lists.
- Public APIs often need an **API key**; without a valid key the server replies with an error like `401 Unauthorised`.
- JSON data is built from objects (like dictionaries), arrays (like lists), strings, numbers, booleans and `null`, nested inside each other.
- `json.loads()` parses a JSON *string* into Python objects; `json.dump()` writes a Python object to a JSON *file*; `json.load()` reads it back.